# AayuSense — Model Training & Evaluation
> Train Random Forest and evaluate on the extracted feature set.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.preprocessing import LabelEncoder

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='darkgrid')
print('Libraries loaded.')

In [ ]:
# Try loading features; fall back to raw data if processed not available
features_path = Path('../data/processed/features.csv')
raw_path      = Path('../data/raw/sample_sensor_data.csv')

if features_path.exists():
    df = pd.read_csv(features_path)
    print('Loaded features CSV:', df.shape)
else:
    df = pd.read_csv(raw_path)
    print('Loaded raw CSV (features not yet generated):', df.shape)

print('Class counts:')
print(df['quality_label'].value_counts())

In [ ]:
SENSOR_COLS = ['ph', 'conductivity', 'turbidity', 'orp']
feature_cols = [c for c in df.columns if c not in ['quality_label', 'timestamp']]

le = LabelEncoder()
y  = le.fit_transform(df['quality_label'])
X  = df[feature_cols].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print('Classes:', le.classes_)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
print('RandomForest training complete.')

In [ ]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
importances = rf.feature_importances_
top_n = min(15, len(feature_cols))
top_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 5))
colours = plt.cm.viridis(np.linspace(0.2, 0.9, top_n))
ax.bar(range(top_n), importances[top_idx], color=colours)
ax.set_xticks(range(top_n))
ax.set_xticklabels([feature_cols[i] for i in top_idx], rotation=45, ha='right')
ax.set_ylabel('Importance Score')
ax.set_title(f'Top {top_n} Feature Importances — Random Forest')
plt.tight_layout()
plt.show()

## Next Steps

- Run `src/train.py` with GridSearchCV for hyper-parameter optimisation
- Compare Random Forest vs XGBoost using `src/evaluation/metrics.py`
- Add the XGBoostModel and run `compare_models()` for a side-by-side table
- Explore SHAP values for model interpretability
- Deploy the best model via the Streamlit dashboard (`dashboard/app.py`)